# 05. What is pressure? Benchmarking tracking pressure metrics against analyst labels

PFF analysts label every pass with the pressure on the passer — `N` none, `P` player pressured,
`A` attempted pressure, `L` passing-lane pressure — and name the pressing player.
This gives ~60k open-play passes of human ground truth.

1. Metrics from tracking (raw positions at the pass frame, smoothed velocities over 0.2 s)
   - nearest-defender distance and the common event-data radius rule (4.57 m)
   - Andrienko et al. (2017) pressure zones
   - time-to-intercept / Pressing Intensity (Bekkers 2025, unravelsports defaults: reaction 0.7 s, vmax 12 m/s, T 1.5 s, σ 0.45)
   - minimum arrival time under the Fujimura–Sugihara motion model (Narizuka et al. 2026)
   - a learned model (LightGBM, game-grouped CV)
2. Benchmark: P vs N, any pressure vs N, A vs N, L vs N
3. Who is the presser?
4. Passing-lane pressure
5. Consequences: which definition predicts pass failure, and does the human label add information?

Metric code: `src/football/pressure.py`. Parameters of the literature metrics are defaults, not tuned.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, cohen_kappa_score, log_loss

from football.common import seg_dist, cv_predict, boot_ci, boot_diff, KEY, GOAL_X
from football.pressure import defender_table, pass_frames, PFF_R_PRESSURED, PFF_R_ATTEMPT
from football.tracking import pre_event_frames
from functools import partial
from concurrent.futures import ProcessPoolExecutor

OUT = os.path.join("..", "outputs")
FIG = os.path.join(OUT, "figs")
os.makedirs(FIG, exist_ok=True)
FPS, LAG = 29.97, 6
LGB_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)

ev = pd.read_csv(os.path.join(OUT, "events.csv.gz"), low_memory=False)
pa = ev.query("event_type == 'PA' and setpiece == 'O' and pressure.notna()", engine="python").copy()
print(f"open-play passes with label: {len(pa):,}")
pa.pressure.value_counts()

open-play passes with label: 61,028


pressure
N    30515
P    22353
L     4637
A     3523
Name: count, dtype: int64

## 1. Positions at the pass frame + velocities
Positions: raw broadcast-tracking coordinates stored in the event file (the frame the analysts coded).
Velocities: difference of the Kalman-**smoothed** coordinates over 0.2 s (raw differences are too noisy).
Smoothed positions themselves lag the raw ones (median 0.65 m, 90th pct 2.6 m) and agree less with the labels.

In [2]:
# smoothed tracking frames at the pass frame and 6 frames (0.2 s) earlier, all passes (cached, ~5 min)
CACHE = os.path.join(OUT, "pass_frames_smoothed.csv.gz")
if not os.path.exists(CACHE):
    all_pa = ev[ev.event_type == "PA"]
    jobs = {g: set(d.pe_id.astype(int)) for g, d in all_pa.groupby("game_id")}
    with ProcessPoolExecutor(6) as ex:
        parts = list(ex.map(partial(pre_event_frames, offsets_frames=(0, LAG), smoothed=True), jobs, jobs.values()))
    pd.DataFrame([r for p in parts for r in p]).to_csv(CACHE, index=False)

# raw positions + velocities; velocity trusted only when both smoothed frames exist and match the raw position (<= 3 m)
f0 = pass_frames(OUT, pa[KEY])
spd = np.hypot(f0.vx, f0.vy)
print(f"velocity trusted for {f0.vel_ok.mean():.1%} of player rows; speed median {spd[f0.vel_ok].median():.2f}, "
      f"99th pct {spd[f0.vel_ok].quantile(.99):.2f} m/s")
frames = dict(tuple(f0.groupby(KEY)))
len(frames)

velocity trusted for 90.7% of player rows; speed median 1.53, 99th pct 5.45 m/s


61028

## 2. Per-pass metrics and per-defender table

In [3]:
ev_rows, def_rows = [], []
for r in pa.itertuples():
    f = frames.get((r.game_id, r.pe_id))
    if f is None:
        continue
    c = f[f.player_id == r.actor_id]
    D = f[~f.atk]
    if c.empty or len(D) < 5:
        continue
    pc, vc = c[["x", "y"]].to_numpy(float)[0], c[["vx", "vy"]].to_numpy(float)[0]
    pdf, vdf = D[["x", "y"]].to_numpy(float), D[["vx", "vy"]].to_numpy(float)
    mates = f[f.atk & (f.player_id != r.actor_id) & (f.position != "GK")][["x", "y"]].to_numpy(float)
    t = defender_table(pc, vc, pdf, vdf, mates)
    i = np.argsort(t["dist"])
    tgt = f[f.player_id == r.target_id]
    if len(tgt):
        q = tgt[["x", "y"]].to_numpy(float)[0]
        ld = seg_dist(pdf, pc, q)
        lane_min, lane_n2, tnear = ld.min(), int((ld < 2).sum()), np.hypot(*(pdf - q).T).min()
        pass_dist, dx = np.hypot(*(q - pc)), q[0] - pc[0]
    else:
        lane_min = lane_n2 = tnear = pass_dist = dx = np.nan
    near10 = np.hypot(D.x - pc[0], D.y - pc[1]).to_numpy() < 10
    ev_rows.append(dict(game_id=r.game_id, pe_id=r.pe_id, label=r.pressure, presser=r.pressure_player_id,
                        complete=r.pass_outcome == "C", non_event=bool(r.non_event),
                        vel_ok_pass=bool(c.vel_ok.iloc[0] and D.vel_ok.to_numpy()[near10].all()),
                        d1=t["dist"][i[0]], d2=t["dist"][i[1]], d3=t["dist"][i[2]],
                        n3=int((t["dist"] < 3).sum()), n5=int((t["dist"] < 5).sum()),
                        andr_sum=t["andr"].sum(), andr_max=t["andr"].max(),
                        andr_goal_sum=t["andr_goal"].sum(), andr_all_sum=t["andr_all"].sum(),
                        pi_max=t["p_int"].max(), pi_comb=1 - np.prod(1 - t["p_int"]),
                        tti_min=t["tti"].min(), arrival_min=t["arrival"].min(),
                        closing1=t["closing"][i[0]], cos1=t["cos_goal"][i[0]], speed1=t["speed"][i[0]],
                        closing_max=t["closing"].max(), carrier_speed=np.hypot(*vc), cx=pc[0], cy=pc[1],
                        lane_min=lane_min, lane_n2=lane_n2, target_near=tnear, pass_dist=pass_dist, dx=dx))
    if r.pressure == "P" and pd.notna(r.pressure_player_id):
        rank = np.empty(len(i)); rank[i] = np.arange(1, len(i) + 1)
        for j, pid in enumerate(D.player_id.to_numpy()):
            def_rows.append(dict(game_id=r.game_id, pe_id=r.pe_id, player_id=pid,
                                 is_presser=pid == r.pressure_player_id, rank_dist=rank[j],
                                 **{k: v[j] for k, v in t.items()}))
E = pd.DataFrame(ev_rows)
Dd = pd.DataFrame(def_rows)
print(f"passes with metrics: {len(E):,}   defender rows (P passes): {len(Dd):,}")
E.groupby("label")[["d1", "n5", "andr_sum", "pi_max", "tti_min", "closing1", "lane_min"]].median().round(2)

passes with metrics: 61,012   defender rows (P passes): 245,777


,d1,n5,andr_sum,pi_max,tti_min,closing1,lane_min
label,,,,,,,
A,3.56,1.0,0.41,0.76,1.21,1.02,2.66
L,3.09,1.0,0.50,0.78,1.19,0.95,2.08
N,6.51,0.0,0.08,0.36,1.65,0.58,4.22
P,2.13,1.0,0.66,0.81,1.14,0.60,1.42


## 3. Benchmark — how well does each metric reproduce the analysts' pressure labels?
Scores are used as-is (sign flipped so that higher = more pressure). The learned model uses all
carrier-centric tracking features (no lane features in the P/A tasks), game-grouped 5-fold CV.

In [4]:
METRICS = {"nearest distance": -E.d1, "defenders < 5 m": E.n5, "Andrienko (sum)": E.andr_sum, "Andrienko (goal only)": E.andr_goal_sum,
           "Andrienko (goal + all mates)": E.andr_all_sum,
           "pressing intensity (max)": E.pi_max, "pressing intensity (Eq. 5 total)": E.pi_comb,
           "time to intercept (min)": -E.tti_min, "min arrival time (FS model)": -E.arrival_min}
CARRIER_FEATS = ["d1", "d2", "d3", "n3", "n5", "andr_sum", "andr_max", "pi_max", "pi_comb", "tti_min",
                 "arrival_min", "closing1", "cos1", "speed1", "closing_max", "carrier_speed", "cx", "cy"]
LANE_FEATS = ["lane_min", "lane_n2", "target_near", "pass_dist", "dx"]
gbm = lambda s: lgb.LGBMClassifier(random_state=s, **LGB_PARAMS)

TASKS = {"P vs N": ["P"], "any (P/A/L) vs N": ["P", "A", "L"], "A vs N": ["A"], "L vs N": ["L"]}
bench, learned = [], {}
for task, pos in TASKS.items():
    m = E.label.isin(pos + ["N"])
    y = E.label[m].isin(pos).astype(int)
    for name, s in METRICS.items():
        bench.append(dict(task=task, metric=name, roc_auc=roc_auc_score(y, s[m]), pr_auc=average_precision_score(y, s[m])))
    feats = CARRIER_FEATS + (LANE_FEATS if task == "L vs N" else [])
    p = cv_predict(gbm, E.loc[m, feats].fillna(-1), y, E.game_id[m], seeds=(0,))
    learned[task] = (m, y, p)
    bench.append(dict(task=task, metric="learned (LightGBM)", roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p)))
    bench.append(dict(task=task, metric="(base rate)", roc_auc=0.5, pr_auc=y.mean()))
bench = pd.DataFrame(bench)
bench.pivot(index="metric", columns="task", values="roc_auc").round(3)

task,A vs N,L vs N,P vs N,any (P/A/L) vs N
metric,,,,
(base rate),0.500,0.500,0.500,0.500
Andrienko (goal + all mates),0.778,0.817,0.868,0.850
Andrienko (goal only),0.756,0.831,0.850,0.836
Andrienko (sum),0.780,0.832,0.864,0.849
defenders < 5 m,0.746,0.773,0.803,0.792
learned (LightGBM),0.848,0.883,0.912,0.891
min arrival time (FS model),0.814,0.824,0.874,0.859
nearest distance,0.792,0.837,0.888,0.869
pressing intensity (Eq. 5 total),0.744,0.749,0.795,0.782


In [5]:
bench.pivot(index="metric", columns="task", values="pr_auc").round(3)

task,A vs N,L vs N,P vs N,any (P/A/L) vs N
metric,,,,
(base rate),0.104,0.132,0.423,0.500
Andrienko (goal + all mates),0.217,0.332,0.809,0.825
Andrienko (goal only),0.217,0.399,0.808,0.827
Andrienko (sum),0.222,0.376,0.815,0.832
defenders < 5 m,0.201,0.269,0.673,0.723
learned (LightGBM),0.411,0.545,0.877,0.881
min arrival time (FS model),0.284,0.366,0.837,0.851
nearest distance,0.228,0.370,0.849,0.856
pressing intensity (Eq. 5 total),0.205,0.267,0.710,0.749


### 3b. Radius rules as binary definitions (P vs N, and any vs N)

In [6]:
rows = []
for task in ["P vs N", "any (P/A/L) vs N"]:
    m = E.label.isin(TASKS[task] + ["N"])
    y = E.label[m].isin(TASKS[task])
    for r_ in [2.0, PFF_R_PRESSURED, 3.0, 4.0, PFF_R_ATTEMPT, 5.0, 6.0]:
        pred = E.d1[m] < r_
        tp = (pred & y).sum()
        rows.append(dict(task=task, radius_m=r_, flagged=pred.mean(), precision=tp / pred.sum(),
                         recall=tp / y.sum(), kappa=cohen_kappa_score(pred, y)))
pd.DataFrame(rows).round(3)

,task,radius_m,flagged,precision,recall,kappa
0,P vs N,2.000,0.211,0.906,0.452,0.448
1,P vs N,2.743,0.328,0.868,0.674,0.618
2,P vs N,3.000,0.363,0.847,0.726,0.642
3,P vs N,4.000,0.476,0.753,0.848,0.633
4,P vs N,4.572,0.533,0.700,0.883,0.586
5,P vs N,5.000,0.571,0.666,0.899,0.544
6,P vs N,6.000,0.650,0.604,0.929,0.450
7,any (P/A/L) vs N,2.000,0.197,0.913,0.361,0.326
8,any (P/A/L) vs N,2.743,0.325,0.885,0.576,0.501
9,any (P/A/L) vs N,3.000,0.367,0.869,0.638,0.542


## 4. Who is the presser?
For `P` passes: is the analyst-named presser the defender ranked first by each metric?
Learned presser model: defender-level LightGBM (game-grouped CV), argmax within each pass.

In [7]:
Dd = Dd.merge(Dd.groupby(KEY).is_presser.any().rename("has_presser").reset_index(), on=KEY)
Dd = Dd[Dd.has_presser].copy()
Dd["neg_dist"], Dd["neg_tti"] = -Dd.dist, -Dd.tti
DEF_FEATS = ["dist", "cos_goal", "closing", "speed", "tti", "p_int", "andr", "arrival", "rank_dist"]
Dd["p_presser"] = cv_predict(gbm, Dd[DEF_FEATS], Dd.is_presser.astype(int), Dd.game_id, seeds=(0,))

top1 = {}
for name, col in [("nearest defender", "neg_dist"), ("min time to intercept", "neg_tti"),
                  ("max Andrienko", "andr"), ("max intercept prob.", "p_int"), ("learned", "p_presser")]:
    best = Dd.loc[Dd.groupby(KEY)[col].idxmax()]
    top1[name] = best.is_presser.mean()
print(f"P passes with presser in frame: {Dd.groupby(KEY).ngroups:,}")
pd.Series(top1, name="top-1 accuracy").round(3)

P passes with presser in frame: 22,231


nearest defender         0.778
min time to intercept    0.640
max Andrienko            0.750
max intercept prob.      0.640
learned                  0.780
Name: top-1 accuracy, dtype: float64

In [8]:
# when the presser is NOT the nearest defender, how do they differ from the nearest one?
pr = Dd[Dd.is_presser]
near = Dd[Dd.rank_dist == 1].set_index(KEY)
cmp_ = pr.set_index(KEY)[["dist", "closing", "cos_goal", "speed", "tti"]].join(
    near[["dist", "closing", "cos_goal", "speed", "tti"]], rsuffix="_nearest")
nn = cmp_[pr.set_index(KEY).rank_dist > 1]
print(f"presser is not the nearest defender: {len(nn) / len(cmp_):.1%}  (n={len(nn):,})")
nn.median().round(2)

presser is not the nearest defender: 22.2%  (n=4,929)


dist                11.34
closing              0.02
cos_goal             0.38
speed                1.63
tti                  2.29
dist_nearest         3.02
closing_nearest      0.03
cos_goal_nearest     0.57
speed_nearest        1.63
tti_nearest          1.38
dtype: float64

## 5. Passing-lane pressure (`L`)
Carrier-centric metrics vs lane features (defenders near the passer→target segment).

In [9]:
m = E.label.isin(["L", "N"]) & E.lane_min.notna()
y = (E.label[m] == "L").astype(int)
rows = [dict(feature=n, roc_auc=roc_auc_score(y, s[m])) for n, s in
        [("nearest distance (carrier)", -E.d1), ("pressing intensity (Eq. 5 total)", E.pi_comb),
         ("lane: min defender distance", -E.lane_min), ("lane: defenders < 2 m", E.lane_n2)]]
pC = cv_predict(gbm, E.loc[m, CARRIER_FEATS].fillna(-1), y, E.game_id[m], seeds=(0,))
pCL = cv_predict(gbm, E.loc[m, CARRIER_FEATS + LANE_FEATS].fillna(-1), y, E.game_id[m], seeds=(0,))
rows += [dict(feature="learned: carrier features", roc_auc=roc_auc_score(y, pC)),
         dict(feature="learned: carrier + lane features", roc_auc=roc_auc_score(y, pCL))]
d, ci, _ = boot_diff(y, pCL, pC, roc_auc_score, n=300, groups=E.game_id[m])
print(f"gain from lane features: {d:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")
pd.DataFrame(rows).round(3)

gain from lane features: 0.006 [0.004, 0.008]


,feature,roc_auc
0,nearest distance (carrier),0.837
1,pressing intensity (Eq. 5 total),0.749
2,lane: min defender distance,0.753
3,lane: defenders < 2 m,0.633
4,learned: carrier features,0.877
5,learned: carrier + lane features,0.883


## 6. Consequences — which definition predicts pass failure?
Open-play passes with a target in frame. Base model: pass geometry (distance, forward component, lane openness,
target's nearest defender, passer location). Add one pressure definition at a time (logistic regression,
game-grouped CV). Then: does the **human** label add information beyond the best tracking definition, and
vice versa?

In [10]:
O = E.dropna(subset=["pass_dist"]).copy()
O["fail"] = (~O.complete).astype(int)
O["h_P"], O["h_A"], O["h_L"] = (O.label == "P").astype(int), (O.label == "A").astype(int), (O.label == "L").astype(int)
# learned P-probability for every pass: fit on P vs N passes of the training games, predict all test passes
from sklearn.model_selection import GroupKFold
O["learned_P"] = 0.0
yP = (O.label == "P").astype(int)
for tr, te in GroupKFold(5).split(O, groups=O.game_id):
    trm = O.index[tr][O.label.iloc[tr].isin(["P", "N"])]
    mdl = gbm(0).fit(O.loc[trm, CARRIER_FEATS].fillna(-1), yP.loc[trm])
    O.loc[O.index[te], "learned_P"] = mdl.predict_proba(O.iloc[te][CARRIER_FEATS].fillna(-1))[:, 1]

BASE = ["pass_dist", "dx", "lane_min", "target_near", "cx", "cy"]
logit = lambda s: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
y = O.fail
p_base = cv_predict(logit, O[BASE], y, O.game_id, seeds=(0,))
defs = {"nearest distance": ["d1"], "4.57 m rule": ["r5"], "Andrienko": ["andr_sum"],
        "pressing intensity": ["pi_comb"], "time to intercept": ["tti_min"], "learned P-prob": ["learned_P"],
        "human label (P/A/L)": ["h_P", "h_A", "h_L"]}
O["r5"] = (O.d1 < PFF_R_ATTEMPT).astype(int)
out, preds = [], {}
for name, cols in defs.items():
    p = cv_predict(logit, O[BASE + cols], y, O.game_id, seeds=(0,))
    preds[name] = p
    d, ci, _ = boot_diff(y, p, p_base, roc_auc_score, n=300, groups=O.game_id)
    out.append(dict(definition=name, roc_auc=roc_auc_score(y, p), log_loss=log_loss(y, p),
                    auc_gain_vs_base=d, ci95=f"[{ci[0]:.4f}, {ci[1]:.4f}]"))
print(f"fail rate {y.mean():.3f};  base ROC-AUC {roc_auc_score(y, p_base):.3f}, log-loss {log_loss(y, p_base):.4f}")
outcome = pd.DataFrame(out).round(4)
outcome

fail rate 0.163;  base ROC-AUC 0.838, log-loss 0.3376


,definition,roc_auc,log_loss,auc_gain_vs_base,ci95
0,nearest distance,0.8406,0.3363,0.0021,"[0.0013, 0.0030]"
1,4.57 m rule,0.8396,0.3368,0.0012,"[0.0007, 0.0016]"
2,Andrienko,0.8397,0.3362,0.0013,"[0.0007, 0.0019]"
3,pressing intensity,0.8396,0.3365,0.0012,"[0.0006, 0.0018]"
4,time to intercept,0.8391,0.3370,0.0007,"[0.0003, 0.0011]"
5,learned P-prob,0.8403,0.3359,0.0018,"[0.0012, 0.0024]"
6,human label (P/A/L),0.8415,0.3350,0.0031,"[0.0024, 0.0038]"


In [11]:
# does the human label add beyond the best tracking definition, and vice versa?
best_track = ["learned_P", "pi_max", "d1"]
p_t = cv_predict(logit, O[BASE + best_track], y, O.game_id, seeds=(0,))
p_th = cv_predict(logit, O[BASE + best_track + ["h_P", "h_A", "h_L"]], y, O.game_id, seeds=(0,))
p_h = preds["human label (P/A/L)"]
d1_, ci1, _ = boot_diff(y, p_th, p_t, roc_auc_score, n=300, groups=O.game_id)
d2_, ci2, _ = boot_diff(y, p_th, p_h, roc_auc_score, n=300, groups=O.game_id)
print(f"human label on top of tracking : +{d1_:.4f} [{ci1[0]:.4f}, {ci1[1]:.4f}]")
print(f"tracking on top of human label : +{d2_:.4f} [{ci2[0]:.4f}, {ci2[1]:.4f}]")
# within deciles of tracking pressure, failure rate by human label
O["track_decile"] = pd.qcut(O.learned_P.rank(method="first"), 5, labels=[f"Q{i}" for i in range(1, 6)])
O.pivot_table(index="track_decile", columns="label", values="fail", aggfunc="mean", observed=True).round(3)

human label on top of tracking : +0.0019 [0.0013, 0.0024]
tracking on top of human label : +0.0011 [0.0005, 0.0017]


label,A,L,N,P
track_decile,,,,
Q1,0.245,0.162,0.064,0.261
Q2,0.152,0.124,0.098,0.246
Q3,0.147,0.129,0.153,0.229
Q4,0.139,0.141,0.215,0.235
Q5,0.094,0.175,0.261,0.250


## 7. Figure — benchmark summary

In [12]:
tab = bench[bench.metric != "(base rate)"].pivot(index="metric", columns="task", values="roc_auc")
tab = tab.sort_values("P vs N")
fig, ax = plt.subplots(figsize=(6.5, 3.8))
tab[["P vs N", "any (P/A/L) vs N", "L vs N"]].plot.barh(ax=ax, width=0.8)
ax.set_xlim(0.5, 1.0); ax.set_xlabel("ROC-AUC vs analyst labels")
ax.set_title("Tracking pressure metrics vs 60k analyst-labelled passes")
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_pressure_benchmark.png"), dpi=200)

## Save

In [13]:
E.to_csv(os.path.join(OUT, "pressure_events.csv.gz"), index=False)
bench.to_csv(os.path.join(OUT, "pressure_benchmark.csv"), index=False)
outcome.to_csv(os.path.join(OUT, "pressure_outcome.csv"), index=False)
print("saved")

saved
